# 23｜葡萄牙污水 MS1峰提取、DDA关联与迭代进样合并

本笔记本使用 pyOpenMS `FeatureFindingMetabo` 对22号的96个mzML逐文件进行MS1色谱峰提取，将DDA-MS2关联到MS1特征，然后对每个站点×批次×离子模式的A/B两次迭代DDA进样进行去冗余合并。

公开数据包没有能由文件名明确识别的blank/QC，因此本步不声称进行了空白扣除。所有MS2（包括未关联事件）都会保留在全局敏感性索引中；只有关联到MS1特征且至少包含3个清洗峰的MS2进入主样品索引。

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd
from IPython.display import display

ROOT = Path(r'E:\\谱图预测PBT_终版')
ENV_DREAMS = Path(r'D:\\Appinstallation\\Anaconda\\envs\\env_dreams\\python.exe')
SRC = ROOT / 'src'
CASE = ROOT / 'external_data/portugal_wastewater_5830725'
MANIFEST_DIR = CASE / 'manifests'
OUT = CASE / '03_feature_extraction_23'
RUN_FEATURE_EXTRACTION = True

assert ENV_DREAMS.is_file(), ENV_DREAMS
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
print('ROOT:', ROOT)
print('Notebook kernel:', sys.executable)
print('env_dreams:', ENV_DREAMS)
print(f'E drive free: {shutil.disk_usage(CASE).free / 1024**3:.1f} GiB')

## 1. 22号门控与软件检查

22号必须已转换并通过96/96个mzML。pyOpenMS原生读取器在Windows上无法直接打开包含中文的路径，主程序会在系统临时目录自动建立一个纯英文NTFS目录联接；它指向原文件，不会复制6 GB mzML。

In [ ]:
nb22_path = MANIFEST_DIR / '22_msconvert_run_manifest.json'
nb22 = json.loads(nb22_path.read_text(encoding='utf-8'))
assert nb22.get('status') == 'READY_FOR_NOTEBOOK_23', nb22.get('status')
assert nb22.get('qc_all_pass') is True

check = subprocess.run(
    [str(ENV_DREAMS), '-c', 'import pyopenms; print(pyopenms.__version__)'],
    check=True, capture_output=True, text=True, env=env,
)
print('✓22号状态:', nb22['status'])
print('✓mzML:', nb22['counts']['passed'], '/', nb22['counts']['planned'])
print('✓pyOpenMS:', check.stdout.strip())

## 2. 冻结参数

参数针对Agilent QTOF质心化数据：MS1质量轨迹10 ppm、峰强阈值200、色谱S/N 3、峰宽2–60 s；DDA-MS2关联20 ppm并结合色谱峰宽；A/B进样合并10 ppm和15 s。参数会写入最终清单并用SHA-256冻结。

In [ ]:
check_params = subprocess.run(
    [str(ENV_DREAMS), '-c',
     'import json; from pbt_msms.portugal_feature_extraction import parameter_payload; print(json.dumps(parameter_payload(), indent=2))'],
    check=True, capture_output=True, text=True, cwd=ROOT, env=env,
)
print(check_params.stdout)

## 3. 逐文件峰提取与A/B合并

程序每完成一个mzML就立即保存特征表、MS2索引、变长峰数组和回执清单。中断后重跑会根据mzML哈希和参数哈希跳过已完成文件。根据小样本试运行，通常约10–30分钟，具体取决于硬盘。

In [ ]:
if RUN_FEATURE_EXTRACTION:
    cmd = [str(ENV_DREAMS), '-m', 'pbt_msms.portugal_feature_extraction',
           '--root', str(ROOT)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)
    print()
    print('✓ Notebook 23 主程序执行完成')
else:
    print('RUN_FEATURE_EXTRACTION=False：没有执行峰提取。')

## 4. 最终QC

必须完成96个进样、48个A/B合并样品单元，且总MS2数必须与22号完全一致。空白扣除为信息性未通过项，不阻断主流程，但必须作为真实案例限制报告。

In [ ]:
manifest_path = MANIFEST_DIR / '23_feature_extraction_run_manifest.json'
run_status_path = MANIFEST_DIR / '23_feature_extraction_run_status.csv'
sample_summary_path = MANIFEST_DIR / '23_sample_consensus_summary.csv'
sample_ms2_path = MANIFEST_DIR / '23_sample_ms2_feature_index.csv.gz'
qc_path = MANIFEST_DIR / '23_feature_extraction_qc_checks.csv'

assert RUN_FEATURE_EXTRACTION, '请将RUN_FEATURE_EXTRACTION设为True后重跑。'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
run_status = pd.read_csv(run_status_path)
sample_summary = pd.read_csv(sample_summary_path)
sample_ms2 = pd.read_csv(sample_ms2_path, low_memory=False)
qc = pd.read_csv(qc_path)

display(pd.Series({
    'status': manifest['status'],
    'runs': manifest['counts']['runs'],
    'sample units': manifest['counts']['sample_units'],
    'MS1 features across runs': manifest['counts']['ms1_features'],
    'MS2 events': manifest['counts']['ms2_events'],
    'linked MS2 fraction': manifest['counts']['linked_ms2_fraction'],
    'model-eligible MS2 fraction': manifest['counts']['model_eligible_ms2_fraction'],
    'sample consensus model-eligible features': manifest['counts']['sample_consensus_model_eligible_ms2_features'],
}, name='value').to_frame())
display(run_status.groupby('mode').agg(
    runs=('dataset_id', 'size'), features=('features_n', 'sum'),
    ms2=('ms2_spectra_n', 'sum'), linked_ms2=('linked_ms2_n', 'sum'),
    eligible_ms2=('model_eligible_ms2_n', 'sum'),
))
display(sample_summary.groupby('mode').agg(
    samples=('sample_id', 'size'), features=('features_total', 'sum'),
    both_injections=('features_detected_both_injections', 'sum'),
    eligible_ms2_features=('model_eligible_ms2_features', 'sum'),
))
display(qc)
assert manifest['status'] == 'READY_FOR_NOTEBOOK_24', manifest['status']
assert manifest['qc_all_required_pass'] is True
assert len(run_status) == 96 and len(sample_summary) == 48 and len(sample_ms2) > 0
print()
print('✓最终状态:', manifest['status'])
print('下一步：24号用冻结模型对真实水样MS/MS进行P/B/T预测与危险性优先排序。')